<div style="display: flex; align-items: center; gap: 18px; margin-bottom: 15px;">
  <img src="https://files.codebasics.io/v3/images/sticky-logo.svg" alt="Codebasics Logo" style="display: inline-block;" width="130">
  <h1 style="font-size: 34px; color: #1f4e79; margin: 0; display: inline-block;">Codebasics Practice Room - Data Engineering Bootcamp </h1>
</div>

## 🧑🏼‍🔧 Setup

In [0]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType, TimestampType
from delta.tables import DeltaTable

In [0]:
# --------------------------------------------
# ⚙️ Databricks Unity Catalog Setup (Auto)
# --------------------------------------------
from pyspark.sql import SparkSession

catalog_name = "practice_db_catalog"
schema_name = "iot_dedup"
volume_name = "data_volume"
folder_name = "iot_dedup"

# 1️⃣ Create Catalog if not exists
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
print(f"✅ Catalog `{catalog_name}` ready.")

# 2️⃣ Create Schema (Database) if not exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
print(f"✅ Schema `{schema_name}` created inside `{catalog_name}`.")

# 3️⃣ Create Volume if not exists
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")
print(f"✅ Volume `{volume_name}` created inside `{catalog_name}.{schema_name}`")

# 4️⃣ Set current context
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE {schema_name}")

# 5️⃣ Define volume-backed paths
base_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/{folder_name}"
raw_path = f"{base_path}/raw"


# 6️⃣ Create directories inside the volume
dbutils.fs.mkdirs(raw_path)

print("✅ Paths initialized successfully:")
print(f"Raw: {raw_path}")



✅ Catalog `practice_db_catalog` ready.
✅ Schema `iot_dedup` created inside `practice_db_catalog`.
✅ Volume `data_volume` created inside `practice_db_catalog.iot_dedup`
✅ Paths initialized successfully:
Raw: /Volumes/practice_db_catalog/iot_dedup/data_volume/iot_dedup/raw


In [0]:
# 🧮 Generate reproducible IoT readings dataset (line-delimited JSON)
import json, random, datetime
from pathlib import Path

random.seed(42)

# Configuration
num_devices = 50
num_records = 700

devices = [f"D{d:03d}" for d in range(1, num_devices + 1)]
statuses = ["normal", "warning", "critical"]

# Starting time
start_time = datetime.datetime(2025, 1, 1, 8, 0, 0)
records = []

# Generate main records
for _ in range(num_records):
    device = random.choice(devices)
    # random intervals 1–15 minutes (to simulate bursty IoT data)
    gap = datetime.timedelta(minutes=random.randint(1, 15))
    start_time += gap
    records.append({
        "device_id": device,
        "timestamp": start_time.strftime("%Y-%m-%dT%H:%M:%S"),
        "temperature": round(random.uniform(18.0, 30.0), 1),
        "status": random.choice(statuses)
    })

# Add artificial near-duplicates (same device, within 3–5 mins)
for _ in range(50):
    r = random.choice(records)
    dup_time = datetime.datetime.strptime(r["timestamp"], "%Y-%m-%dT%H:%M:%S") \
        + datetime.timedelta(minutes=random.randint(1, 4))
    records.append({
        "device_id": r["device_id"],
        "timestamp": dup_time.strftime("%Y-%m-%dT%H:%M:%S"),
        "temperature": r["temperature"],
        "status": r["status"]
    })

# Output path in Unity Catalog Volume
out_path = "/Volumes/practice_db_catalog/iot_dedup/data_volume/iot_dedup/raw/iot_readings.json"
Path("/Volumes/practice_db_catalog/iot_dedup/data_volume/iot_dedup/raw").mkdir(parents=True, exist_ok=True)

# Write records to the output file
with open(out_path, "w") as f:
    for row in records:
        json.dump(row, f)
        f.write("\n")

print(f"✅ Generated {len(records)} IoT readings across {len(devices)} devices at {out_path}")


✅ Generated 750 IoT readings across 50 devices at /Volumes/practice_db_catalog/iot_dedup/data_volume/iot_dedup/raw/iot_readings.json



# ❓ Scenario Question: IoT — Dedup by Watermark (PySpark) [Easy]




## 🗂️ Scenario

You are working with **IoT sensor data** streaming in from multiple devices.  
Due to network latency and retries, the system occasionally receives **duplicate readings** within a short interval.  
Your goal is to **remove duplicates** that occur within a **5-minute watermark window** for each `device_id`.

The result should keep **only the earliest event** per device within each 5-minute time span.

---

## 🎯 Task

Perform the following transformations using PySpark:

1. **Read** the input data from `/Volumes/practice_db_catalog/iot_dedup/data_volume/iot_dedup/raw/iot_readings.json`.  
2. **Parse** the `timestamp` column as Spark `timestamp`.  
3. **Deduplicate** records within a **5-minute sliding window** per device.  
4. **Keep** only the earliest event (lowest timestamp) in each 5-minute window.  
5. **Display** the final cleaned DataFrame (`df_final`) without duplicates.

---

## 🧩 Assumptions

- Input file is line-delimited JSON located at `/Volumes/practice_db_catalog/iot_dedup/data_volume/iot_dedup/raw/iot_readings.json`.  
- Each record contains:  
  `device_id`, `timestamp`, `temperature`, and `status`.  
- Duplicates may occur when two or more events for the same device are received within 5 minutes.  
- Timestamps are in ISO 8601 format (e.g., `2025-01-02T10:15:30`).  

---

## 📦 Deliverables

- **Output Format:** Spark DataFrame (displayed via `.show()`)  
- **Expected Columns:**  
  `device_id`, `timestamp`, `temperature`, `status`

---

## 🧠 Notes

- Use window functions (`Window.partitionBy().orderBy()`) and `lag()` to compare timestamps.  
- To simulate a watermark logic, consider filtering out events where  
  `timestamp - prev_timestamp < 5 minutes`.  
- Use `dropDuplicates()` carefully — it only removes exact duplicates, not time-based overlaps.  
- Think of it as **deduplication by time proximity**, not by identical records.

---

## 🧩 Example Output (simplified)

| device_id | timestamp           | temperature | status  |
|------------|--------------------|--------------|----------|
| D001       | 2025-01-01T10:00:00 | 23.5         | normal  |
| D001       | 2025-01-01T10:06:00 | 23.8         | normal  |
| D002       | 2025-01-01T10:02:00 | 19.7         | warning |
| D002       | 2025-01-01T10:08:00 | 19.9         | normal  |

---

## 🛢️Input data

In [0]:
df_raw = spark.read.json("/Volumes/practice_db_catalog/iot_dedup/data_volume/iot_dedup/raw/iot_readings.json")
display(df_raw.limit(5))

device_id,status,temperature,timestamp
D041,warning,18.3,2025-01-01T08:02:00
D016,normal,19.7,2025-01-01T08:06:00
D044,normal,28.7,2025-01-01T08:18:00
D038,normal,18.4,2025-01-01T08:25:00
D014,normal,24.1,2025-01-01T08:29:00


# 📝 Your Solution

In [0]:
# ✍️ Your Solution Here

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Steps:
# 1. Read the JSON file
df_raw = spark.read.json("/Volumes/practice_db_catalog/iot_dedup/data_volume/iot_dedup/raw/iot_readings.json")
# display(df_raw.limit(5))

# 2. Convert timestamp column to Spark timestamp
df_raw = df_raw.withColumn("timestamp", F.to_timestamp("timestamp"))
# display(df_raw.limit(5))

# 3. Use window to compare with previous reading per device
device_window = Window.partitionBy("device_id").orderBy("timestamp")

df_gap = df_raw.withColumn(
    "prev_timestamp", F.lag("timestamp").over(device_window)).withColumn(
        "gap_mins", (F.unix_timestamp("timestamp")- F.unix_timestamp("prev_timestamp"))/ 60)
    
# display(df_gap.limit(5))
# 4. Filter out duplicates within 5-minute difference
df_flag = df_gap.withColumn(
    "is_new_window",
    F.when((F.col("prev_timestamp").isNull()) | (F.col("gap_mins") >= 5),1).otherwise(0))

# display(df_flag.limit(5))   

# 5. Return df_final
df_final = df_flag.filter("is_new_window= 1").select("device_id","timestamp","temperature","status")
display(df_final.limit(5))

device_id,timestamp,temperature,status
D001,2025-01-01T13:22:00.000Z,26.6,critical
D001,2025-01-01T14:24:00.000Z,21.9,normal
D001,2025-01-01T17:03:00.000Z,29.1,normal
D001,2025-01-01T17:32:00.000Z,21.2,warning
D001,2025-01-01T19:36:00.000Z,25.5,critical


## 🔍 Validation Questions

After creating the final DataFrame (`df_final`), answer these to check your understanding:

1. How many **total records** were present before and after deduplication?  
2. Which **device** had the most duplicate readings removed?  
3. What is the **time range (min → max)** of readings for device `D010`?  
4. How many unique **devices** are present in the dataset?  
5. For device `D025`, list all remaining timestamps after deduplication.

---

In [0]:
# How many total records were present before and after deduplication?

bef_dedup = df_raw.count()
print(f"Total records present before deduplication: {bef_dedup}")

aft_dedup = df_final.count()
print(f"Total records present after deduplication: {aft_dedup}")

Total records present before deduplication: 750
Total records present after deduplication: 696


In [0]:
# Which device had the most duplicate readings removed?
df_before = df_raw.groupBy("device_id").count().withColumnRenamed("count", "count_before")
df_after = df_final.groupBy("device_id").count().withColumnRenamed("count", "count_after")

df_removed = df_before.join(df_after, "device_id") \
    .withColumn("duplicates_removed", F.col("count_before") - F.col("count_after")) \
    .orderBy(F.col("duplicates_removed").desc())

display(df_removed.select("device_id", "duplicates_removed").limit(1))

device_id,duplicates_removed
D026,7


In [0]:
# What is the time range (min → max) of readings for device D010?
df_d010 = df_final.filter(F.col("device_id") == "D010")
df_time_range = df_d010.agg(
    F.min("timestamp").alias("min_timestamp"),
    F.max("timestamp").alias("max_timestamp")
)
display(df_time_range)

min_timestamp,max_timestamp
2025-01-01T12:29:00.000Z,2025-01-04T13:14:00.000Z


In [0]:
# How many unique devices are present in the dataset?

device_unique = df_final.select("device_id").distinct().count()
print(f"Total unique devices: {device_unique}")

Total unique devices: 50


In [0]:
# For device D025, list all remaining timestamps after deduplication.
df_d025 = df_final.filter(F.col("device_id") == "D025").select("timestamp").orderBy("timestamp")
display(df_d025)

timestamp
2025-01-01T09:15:00.000Z
2025-01-02T06:06:00.000Z
2025-01-02T21:38:00.000Z
2025-01-02T23:36:00.000Z
2025-01-03T11:04:00.000Z
2025-01-04T05:38:00.000Z
2025-01-04T08:45:00.000Z
2025-01-05T03:56:00.000Z
